# 01 — Synthetic Barrel Data Generation

This notebook generates and visualizes **synthetic barrel point clouds** used to
train the learned cleanup models (GridUNet denoiser and PointNet classifier).

The synthetic generator (`barrel_synth.py`) produces barrels with:
- Realistic geometry calibrated from real `.obscan` scan statistics
- Injected artifacts: bung holes, floater clusters, head-pole dropout
- Gaussian radial noise matching real-scan noise floor (~1.5 mm RMS)

**Sections:**
1. Setup & Imports
2. Generate a Single Barrel
3. Visualize the Point Cloud
4. Inspect Noise & Artifacts
5. Batch Generation Statistics
6. Export to `.npz`

## 1. Setup & Imports

In [ ]:
import sys

# 1. Install scipy and other common dependencies for reconstruction projects
!{sys.executable} -m pip install scipy scikit-image opencv-python tqdm

import os

# 2. Add reconstruction package to path
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')),
                                'Dual-Axis-Pass-Through-Barrel-Scanner', 'reconstruction'))
# Also try relative path (works when running from repo root)
sys.path.insert(0, os.path.join('..', 'reconstruction'))

# 3. Now import the libraries
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from barrel_synth import generate_barrel, generate_gt_grid, generate_barrel_params
from barrel_reconstruct import N_AZ

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100
print('Imports OK')

  Using cached numpy-2.5.1-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pillow-12.3.0-cp314-cp314-win_amd64.whl.metadata (9.3 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached numpy-2.5.1-cp314-cp314-win_amd64.whl (12.6 MB)
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ------------- -------------------------- 3.1/9.5 MB 15.8 MB/s eta 0:00:01
   ---------------------------- ----------- 6.8/9.5 MB 17.1 MB/s eta 0:00:01
   ---------------------------------------- 9.5/9.5 MB 17.0 MB/s  0:00:00
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 2.3/2.3 MB 17.5 MB/s  0:00:00
Using cached pillow-12.3.0-cp314-cp314-win_amd64.whl (7.2 MB)
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)

   ---------------------------------------- 0/8 [pypar

ModuleNotFoundError: No module named 'scipy'

## 2. Generate a Single Barrel

The `generate_barrel()` function returns a dict with clean and noisy point clouds,
per-point labels, and barrel geometry parameters.

In [ ]:
b = generate_barrel(seed=42, n_points=200_000,
                    add_bung=True, add_floaters=True,
                    head_dropout_rate=0.3)

P_clean = b['P_clean']
P_noisy = b['P_noisy']
N_noisy = b['N_noisy']
labels  = b['labels']
params  = b['params']

print(f"Clean points: {len(P_clean):,}")
print(f"Noisy points: {len(P_noisy):,} (includes floaters)")
print(f"Barrel span:  {params['span_m']*1000:.1f} mm")
print(f"Bilge radius: {params['r_bilge_m']*1000:.1f} mm")
print(f"Crozehead:    {params['r_crozehead_m']*1000:.1f} mm")
print(f"\nLabel counts:")
for lbl, name in [(0,'clean'), (1,'bung'), (2,'floater'), (3,'head dropout')]:
    print(f"  {name}: {(labels==lbl).sum():,}")

## 3. Visualize the Point Cloud

Subsample for plotting speed. Color by label: blue=clean, red=bung, yellow=floater.

In [ ]:
# Subsample for 3D scatter
rng = np.random.default_rng(0)
n_show = min(20_000, len(P_noisy))
idx = rng.choice(len(P_noisy), n_show, replace=False)

colors = np.array(['#3498db','#e74c3c','#f1c40f','#95a5a6'])
c = colors[labels[idx]]

fig = plt.figure(figsize=(14, 6))

# Side view (Y-Z)
ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(P_noisy[idx,0], P_noisy[idx,1], P_noisy[idx,2],
            c=c, s=0.3, alpha=0.6)
ax1.set_xlabel('X (axis)'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')
ax1.set_title('Noisy Point Cloud (3D)')
ax1.view_init(elev=20, azim=45)

# Cross-section at midplane
mid_mask = np.abs(P_noisy[:,0]) < 0.01
ax2 = fig.add_subplot(122)
ax2.scatter(P_noisy[mid_mask,1], P_noisy[mid_mask,2], s=0.5, alpha=0.3, c='#3498db')
ax2.set_aspect('equal')
ax2.set_xlabel('Y (m)'); ax2.set_ylabel('Z (m)')
ax2.set_title('Midplane Cross-Section (|x| < 10mm)')

plt.tight_layout()
plt.show()

## 4. Inspect Noise & Artifacts

Compare clean vs noisy radial distributions and visualize bung/floater locations.

In [ ]:
rho_clean = np.linalg.norm(P_clean, axis=1) * 1000  # mm
rho_noisy = np.linalg.norm(P_noisy[:len(P_clean)], axis=1) * 1000
residual = rho_noisy - rho_clean

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Radial distribution
axes[0].hist(rho_clean, bins=100, alpha=0.5, label='Clean', color='#3498db')
axes[0].hist(rho_noisy, bins=100, alpha=0.5, label='Noisy', color='#e74c3c')
axes[0].set_xlabel('Radial distance (mm)')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].set_title('Radial Distance Distribution')

# Noise residual
axes[1].hist(residual, bins=100, color='#2ecc71', alpha=0.7)
axes[1].axvline(0, color='k', linestyle='--', linewidth=0.5)
axes[1].set_xlabel('Radial residual (mm)')
axes[1].set_title(f'Noise Residual (RMS={np.sqrt((residual**2).mean()):.2f} mm)')

# Label breakdown pie
label_counts = [(labels==i).sum() for i in range(4)]
label_names = ['Clean', 'Bung', 'Floater', 'Head Dropout']
nonzero = [(n,c) for n,c in zip(label_names, label_counts) if c > 0]
axes[2].pie([c for _,c in nonzero], labels=[n for n,_ in nonzero],
            autopct='%1.1f%%', colors=['#3498db','#e74c3c','#f1c40f','#95a5a6'][:len(nonzero)])
axes[2].set_title('Point Label Distribution')

plt.tight_layout()
plt.show()

## 5. Batch Generation Statistics

Generate multiple barrels and summarize the range of parameters & artifact rates.

In [ ]:
n_barrels = 10
stats = {'span_mm': [], 'bilge_mm': [], 'rho_mean': [], 'rho_std': [],
         'has_bung': [], 'n_floaters': []}

for i in range(n_barrels):
    b = generate_barrel(seed=i, n_points=100_000)
    p = b['params']
    stats['span_mm'].append(p['span_m'] * 1000)
    stats['bilge_mm'].append(p['r_bilge_m'] * 1000)
    rho = b['rho_gt'] * 1000
    stats['rho_mean'].append(rho.mean())
    stats['rho_std'].append(rho.std())
    stats['has_bung'].append((b['labels']==1).sum() > 0)
    stats['n_floaters'].append((b['labels']==2).sum())
    print(f"  barrel {i:2d}: span={p['span_m']*1000:.1f}mm, "
          f"bilge={p['r_bilge_m']*1000:.1f}mm, "
          f"bung={'yes' if stats['has_bung'][-1] else 'no':>3s}, "
          f"floaters={stats['n_floaters'][-1]}")

print(f"\n--- Summary across {n_barrels} barrels ---")
print(f"  Span:      {np.mean(stats['span_mm']):.1f} ± {np.std(stats['span_mm']):.1f} mm")
print(f"  Bilge:     {np.mean(stats['bilge_mm']):.1f} ± {np.std(stats['bilge_mm']):.1f} mm")
print(f"  Bung rate: {sum(stats['has_bung'])}/{n_barrels}")
print(f"  Floaters:  {np.mean(stats['n_floaters']):.0f} ± {np.std(stats['n_floaters']):.0f}")

## 6. Export to `.npz`

Save a synthetic barrel to disk for offline training or debugging.

In [ ]:
b = generate_barrel(seed=42, n_points=500_000)
save_params = {k: v for k, v in b['params'].items() if k != 'rng'}

out_path = os.path.join('..', 'models', 'synthetic_barrel_42.npz')
np.savez_compressed(out_path,
    P_clean=b['P_clean'], N_clean=b['N_clean'],
    rho_gt=b['rho_gt'], el_clean=b['el_clean'],
    az_clean=b['az_clean'], zone=b['zone'],
    P_noisy=b['P_noisy'], N_noisy=b['N_noisy'],
    labels=b['labels'],
    **{f'param_{k}': v for k, v in save_params.items()})

print(f"Saved to {out_path}")
print(f"File size: {os.path.getsize(out_path) / 1e6:.1f} MB")